# Getting started with `processtensor`

`processtensor` is a minimal numerical implementation of **process tensors**—the general description of multi-time quantum processes with non-Markovian memory—along with the quantum states and channels they act on.

A *k-slot process tensor* is a multilinear map taking an initial system state $\rho_\mathrm{in}$ and a sequence of $k$ *instruments* $\mathbf{A}_{0:k-1}$ (CP maps applied to the system at intermediate times) to the output state:

$$\mathcal{T}_{0:k}[\rho_\mathrm{in}, \mathbf{A}_{0:k-1}] = \rho_\mathrm{out}.$$

Like a quantum channel, it admits several equivalent representations; this package works with the **Choi representation** $\Upsilon_{0:k}$, a positive semidefinite operator on the in/out spaces of every time step that obeys causal (containment) constraints.

This notebook walks through the three core objects—`QuantumState`, `QuantumChannel`, `ProcessTensor`—and the temporal correlation measures. Install with `pip install -e .` from the repository root (add `'.[examples]'` for matplotlib, used in the final section).

In [ ]:
import numpy as np

from processtensor import (
    ProcessTensor,
    QuantumChannel,
    QuantumState,
    bell_state,
    pauli,
    unitary_from_hamiltonian,
)

## 1. Quantum states

`QuantumState` stores a (possibly multipartite) density matrix as a tensor with one fused Liouville leg per subsystem. Build one from a density matrix or a pure state vector:

In [ ]:
bell = QuantumState.from_matrix(bell_state(), dims=(2, 2))  # two-qubit Bell pair

print(bell)
print("valid: ", bell.is_valid())
print("purity:", bell.purity())
print("entropy:", bell.entropy())

Subsystem structure gives access to marginals and entanglement measures. For the Bell pair, each marginal is maximally mixed, the mutual information is maximal ($2\ln 2$), and the negativity across the middle cut is $1/2$:

In [ ]:
marginal = bell.partial_trace([0])  # keep subsystem 0
print("marginal:\n", marginal.choi.real)

# Defaults: each subsystem its own block / a half cut (= negativity([0])).
print("mutual information:", bell.mutual_information())
print("negativity:", bell.negativity())

## 2. Quantum channels

`QuantumChannel` stores a CP map as its superoperator with legs `(in, out)`. Construct one from Kraus operators, a unitary, or a Stinespring dilation $\mathrm{Tr}_E[U(\rho\otimes\sigma_E)U^\dagger]$. Pauli matrices come from `pauli`, which also accepts multi-character strings: `pauli("XZ")` is $X \otimes Z$.

In [ ]:
p = 0.75  # depolarizing probability: rho -> (1 - p) rho + p I/2
depolarize = QuantumChannel.from_kraus(
    [np.sqrt(1 - 3 * p / 4) * pauli("I")] + [np.sqrt(p / 4) * pauli(s) for s in "XYZ"]
)
print("CPTP:", depolarize.is_cptp())

rho = QuantumState.from_pure(np.array([1, 0]))  # |0><0|
print("depolarized |0><0|:\n", depolarize.apply(rho).density_matrix.real)

## 3. Process tensors from system-environment dynamics

Any process tensor arises physically as the multi-time dynamics of a system coupled to an environment: a sequence of joint unitaries interleaved with the instrument slots, with the environment traced out at the end (a multi-time Stinespring dilation). `ProcessTensor.from_stinespring` implements exactly this.

As a worked example, take a qubit exchanging information with a single environment qubit under a Heisenberg interaction $H = -\tfrac{\theta}{2}(XX + YY + ZZ)$. At $\theta = \pi/2$ the joint unitary is a SWAP (up to phase), so the process is a perfect one-step quantum memory:

In [ ]:
def heisenberg(theta):
    return -theta / 2 * (pauli("XX") + pauli("YY") + pauli("ZZ"))


U = unitary_from_hamiltonian(heisenberg(np.pi / 2))  # SWAP-like
plus = np.ones((2, 2)) / 2  # environment in |+><+|

pt = ProcessTensor.from_stinespring([U, U], plus)  # 1-slot process tensor
print(pt)
print("slots:", pt.k)
print("valid:", pt.is_valid())  # completely positive + causally ordered

`apply` is the multilinear action of the process tensor: it contracts an initial state and a sequence of CP instruments to the output state. The SWAP process stores the input in the environment and releases it one step later, so the final output equals the initial state *no matter what* trace-preserving instrument acts in between:

In [ ]:
rho_in = np.array([[0.8, 0.1j], [-0.1j, 0.2]])

for name, instrument in [
    ("X gate", QuantumChannel.from_unitary(pauli("X"))),
    ("depolarize", depolarize),
]:
    rho_out = pt.apply(rho_in, [instrument])
    print(f"{name}: output == input? {np.allclose(rho_out.density_matrix, rho_in)}")

## 4. Temporal correlation measures

Two measures quantify the memory carried by the environment:

- **Generalized quantum mutual information (GQMI)** $\mathcal{I}[\Upsilon] = S(\Upsilon \| \Upsilon_\mathrm{Markov})$ – the relative entropy between the process and the product of its single-time-step marginals (the closest Markovian process). It captures *all* temporal correlations, classical and quantum.
- **Temporal negativity** – entanglement negativity across a temporal cut of the Choi state. It is nonzero only for *genuinely quantum* temporal correlations (temporal entanglement).

The SWAP process is maximally correlated in both senses. Contrast it with a controlled-X interaction $H = -\tfrac{\theta}{2}(XI - XZ)$ at $\theta = \pi/2$: with the environment in $|+\rangle$, it applies a perfectly *classically* correlated bit flip at both steps—half the GQMI, but zero temporal entanglement:

In [ ]:
def controlled_x(theta):
    return -theta / 2 * (pauli("XI") - pauli("XZ"))


U_cx = unitary_from_hamiltonian(controlled_x(np.pi / 2))
pt_cx = ProcessTensor.from_stinespring([U_cx, U_cx], plus)

print("              GQMI    temporal negativity")
for name, p in [("Heisenberg:  ", pt), ("Controlled-X:", pt_cx)]:
    print(f"{name} {p.gqmi():.4f}  {p.temporal_negativity():.4f}")
print(f"(ln 2 = {np.log(2):.4f}, 2 ln 2 = {2 * np.log(2):.4f})")

## 5. Object-agnostic measures

The quantities above are not special to any one object. The `processtensor.measures` module (re-exported at the package top level) defines them as functions of *any* leg-based quantum tensor through its Choi representation—the same `negativity` computes the spatial entanglement of a density matrix or the temporal entanglement of a process tensor, and `entropy`, `mutual_information`, `purity`, and `choi_matrix` work the same way:

In [ ]:
from processtensor import entropy, negativity

print("state negativity:  ", negativity(bell, [0]))  # == bell.negativity()
print("process negativity:", negativity(pt, [0, 1]))  # == pt.temporal_negativity()
print("channel Choi entropy:", entropy(depolarize))

## 6. The link product

The literature's standard way to compose objects in the Choi picture is the *link product* (Chiribella, D'Ariano & Perinotti): for Choi operators sharing a space $X$,

$$\Upsilon_1 \star \Upsilon_2 = \mathrm{Tr}_X\!\left[(\Upsilon_1 \otimes \mathbb{I})\,(\mathbb{I} \otimes \Upsilon_2^{T_X})\right].$$

In the fused-leg convention this is simply a tensor contraction over the linked legs—the partial transpose is absorbed by the leg fusion. Feeding a state through a channel, composing channels, and applying instruments to a process tensor are all special cases:

In [ ]:
from processtensor import QTensor, link_product
from processtensor.utils import rft_unitary

# Compose channels: link the out leg of the first to the in leg of the second.
x_gate = QuantumChannel.from_unitary(pauli("X"))
linked = link_product(x_gate, depolarize, [1], [0])
print("matches compose():", np.allclose(linked.choi, depolarize.compose(x_gate).choi))

The link product also rebuilds a process tensor from its basic building blocks. Wrap each system-environment unitary as a 4-leg tensor with legs `(env_in, sys_in, env_out, sys_out)`, then thread the environment through the chain—environment state in, link each `env_out` to the next `env_in`, and trace the final environment leg:

In [ ]:
W = QTensor(rft_unitary(U, 2, 2), (2, 2, 2, 2))  # (env_in, sys_in, env_out, sys_out)
sigma = QuantumState.from_matrix(plus)  # initial environment state

t = link_product(sigma, W, [0], [0])  # legs (in0, env, out0)
t = link_product(t, W, [1], [0])  # legs (in0, out0, in1, env, out1)
t = t.partial_trace([0, 1, 2, 4])  # trace out the final environment leg
print("matches from_stinespring:", np.allclose(t.data, pt.data))

## 7. Bond entropy and Schmidt rank

Across any contiguous cut, a quantum tensor admits an *operator Schmidt decomposition* of its Choi state, $\Upsilon = \sum_i s_i\, A_i \otimes B_i$. Two derived quantities:

- **Schmidt rank** – the number of nonzero coefficients. For a process tensor this equals the minimal bond dimension of an MPO representation at that temporal cut (at most $d_E^2$ for an environment of dimension $d_E$).
- **Bond entropy** – the entropy of the normalized squared coefficients (the operator entanglement entropy), measuring how compressible the object is across the cut.

The SWAP process saturates the $d_E^2 = 4$ bound with maximal bond entropy $\ln 4$, while the controlled-X process needs a smaller bond. (Note these are operator-space quantities: a *pure state* whose state vector has Schmidt rank $r$ has operator Schmidt rank $r^2$.)

In [ ]:
for name, p in [("Heisenberg (SWAP)", pt), ("Controlled-X     ", pt_cx)]:
    print(f"{name}: rank = {p.schmidt_rank()}, bond entropy = {p.bond_entropy():.4f}")
print(f"(ln 4 = {np.log(4):.4f}; dE^2 = 4 bounds the rank / MPO bond dimension)")

## 8. Sweeping the interaction strength

Finally, sweep $\theta$ for both models and plot the two measures (requires matplotlib):

In [ ]:
import matplotlib.pyplot as plt

thetas = np.linspace(0, 2 * np.pi, 101)
models = {"Heisenberg": heisenberg, "Controlled-X": controlled_x}

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for ax, (name, hamiltonian) in zip(axes, models.items()):
    pts = [
        ProcessTensor.from_stinespring([unitary_from_hamiltonian(hamiltonian(t))] * 2, plus)
        for t in thetas
    ]
    ax.plot(thetas, [p.gqmi() for p in pts], label="GQMI")
    negs = [p.temporal_negativity() for p in pts]
    ax.plot(thetas, negs, "--", label="Temporal negativity")
    ax.set_title(name)
    ax.set_xlabel(r"$\theta$")
    ax.set_xticks([0, np.pi, 2 * np.pi], ["0", r"$\pi$", r"$2\pi$"])
axes[0].set_ylabel("Temporal correlations")
axes[0].legend()
fig.tight_layout()

The Heisenberg model develops temporal *entanglement* (nonzero negativity) peaking at the SWAP points, while the controlled-X model builds purely classical memory: its GQMI oscillates but its temporal negativity is identically zero.

## Where to go next

- `DESIGN.md` – design philosophy and the full mathematical conventions
  (vectorization, leg ordering, Choi normalization).
- `processtensor.measures`, `.operations`, `.representations` – the
  object-agnostic functions behind the methods used above.
- `tests/` – compact worked examples with analytically known values.